# Notebook 02 — A/B Testing: Experiment Design & Results

## What is A/B Testing?

**A/B testing** is an *experimental design framework* where you:
1. Split a population into two groups — **A (Control)** and **B (Treatment)**
2. Apply a different condition to each group
3. Measure an outcome metric for both groups
4. Compare the results to quantify the **effect of your treatment**

The goal of A/B testing is to **measure what happened** — it answers the question:
> *"Did group B perform differently than group A, and by how much?"*

**A/B testing is NOT the same as Hypothesis Testing.** A/B testing is the experiment — hypothesis testing (Notebook 03) is the statistical tool used to determine whether the observed difference is real or due to random chance.

---

## Experiment Design for This Dataset

| | Group | Definition |
|---|---|---|
| **A — Control** | Organic | Customers who *received* an offer but did **not view** it. They behave as if no offer exists. |
| **B — Treatment** | Exposed | Customers who *viewed* the offer. They were aware of the promotion. |

**Why this split?** The Starbucks dataset does not have a pure holdout group (customers who received no offers). The closest proxy for organic behavior is customers who received an offer but never opened it.

**Metric:** Offer completion rate (conversion)

---

## Power Analysis — Are We Sufficiently Powered?

Before running any test, we check whether our sample size is large enough to reliably detect a meaningful effect.

In [1]:
import sys
from pathlib import Path
_root = Path().resolve(); _root = _root.parent if _root.name == 'notebooks' else _root; sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.extract import extract
from src.transform import transform
from src.constants import REPORTS_FIGURES, ALPHA, POWER
from src.utils.stats import required_sample_size
from src.utils.plot import save_fig, bar_comparison, funnel_chart

REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

portfolio, profile, transcript = extract()
tables = transform(portfolio, profile, transcript)
master = tables['master_table']
print(f'Master table loaded: {master.shape}')

Master table loaded: (115609, 20)


## 1. Power Analysis

**Minimum Detectable Effect (MDE):** The smallest effect size we want to be able to detect. We use 5 percentage points as the threshold — smaller effects are not business-relevant.

- **α (alpha)** = 0.05 → 5% chance of a false positive (Type I error)
- **Power** = 0.80 → 80% chance of detecting a real effect (1 - Type II error)

In [2]:
baseline = master['completed'].mean()
mde = 0.05
n_needed = required_sample_size(baseline, mde, alpha=ALPHA, power=POWER)

control_n   = len(master[master['viewed'] == False])
treatment_n = len(master[master['viewed'] == True])

print(f'Overall baseline conversion rate : {baseline:.1%}')
print(f'MDE (minimum detectable effect)  : {mde:.0%} pp absolute')
print(f'Required sample size per group   : {n_needed:,}')
print()
print(f'Control group size   : {control_n:,}  → {"SUFFICIENT" if control_n >= n_needed else "INSUFFICIENT"}')
print(f'Treatment group size : {treatment_n:,}  → {"SUFFICIENT" if treatment_n >= n_needed else "INSUFFICIENT"}')

Overall baseline conversion rate : 58.3%
MDE (minimum detectable effect)  : 5% pp absolute
Required sample size per group   : 1,178

Control group size   : 16,676  → SUFFICIENT
Treatment group size : 98,933  → SUFFICIENT


## 2. Group Definitions & Conversion Rates

We measure conversion rate per group for each offer type.

In [3]:
# Overall A vs B
group_summary = master.groupby('viewed').agg(
    n_customers=('person', 'count'),
    n_completed=('completed', 'sum'),
).assign(conversion_rate=lambda x: x['n_completed'] / x['n_customers'])
group_summary.index = ['A — Control (not viewed)', 'B — Treatment (viewed)']
group_summary

,n_customers,n_completed,conversion_rate
A — Control (not viewed),16676,5896,0.353562
B — Treatment (viewed),98933,61501,0.621643


In [4]:
# By offer type
type_summary = master.groupby(['offer_type', 'viewed']).agg(
    n=('person', 'count'),
    conversions=('completed', 'sum'),
).assign(rate=lambda x: (x['conversions'] / x['n'] * 100).round(1))
type_summary.index = type_summary.index.set_levels(['A: Not Viewed', 'B: Viewed'], level=1)
type_summary

n  conversions  rate
offer_type    viewed                                 
bogo          A: Not Viewed   4385         2204  50.3
              B: Viewed      44223        29763  67.3
discount      A: Not Viewed   8551         3692  43.2
              B: Viewed      39815        31738  79.7
informational A: Not Viewed   3740            0   0.0
              B: Viewed      14895            0   0.0

In [5]:
# Visual comparison
pivot = type_summary['rate'].unstack(level=1)
ax = pivot.plot(kind='bar', figsize=(9, 5), color=['#C0C0C0', '#00704A'], edgecolor='white')
ax.set_title('Conversion Rate — Control (A) vs Treatment (B) by Offer Type')
ax.set_ylabel('Conversion Rate (%)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Group')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

fig = ax.get_figure()
fig.tight_layout()
save_fig(fig, REPORTS_FIGURES / '02_ab_conversion_by_type.png')
plt.show()

## 3. Offer Funnel by Type

The funnel shows drop-off at each step for each offer type.

In [6]:
funnel_data = master.groupby('offer_type').agg(
    received=('person', 'count'),
    viewed=('viewed', 'sum'),
    completed=('completed', 'sum'),
).assign(
    view_rate=lambda x: (x['viewed'] / x['received'] * 100).round(1),
    complete_rate=lambda x: (x['completed'] / x['received'] * 100).round(1),
)
funnel_data

,received,viewed,completed,view_rate,complete_rate
offer_type,,,,,
bogo,48608,44223,31967,91.0,65.8
discount,48366,39815,35430,82.3,73.3
informational,18635,14895,0,79.9,0.0


## 4. Lift — Raw Observed Effect

**Lift** = the raw observed difference between treatment and control, before any statistical validation.

This is the A/B testing result — we observed it. Whether it's statistically significant is answered in Notebook 03.

In [7]:
results = []
for offer_type in master['offer_type'].dropna().unique():
    grp = master[master['offer_type'] == offer_type]
    ctrl_rate  = grp[grp['viewed'] == False]['completed'].mean()
    trt_rate   = grp[grp['viewed'] == True]['completed'].mean()
    lift_abs   = trt_rate - ctrl_rate
    lift_rel   = lift_abs / ctrl_rate * 100 if ctrl_rate > 0 else np.nan
    results.append({
        'offer_type': offer_type,
        'control_rate': f'{ctrl_rate:.1%}',
        'treatment_rate': f'{trt_rate:.1%}',
        'lift_absolute': f'{lift_abs:+.1%}',
        'lift_relative': f'{lift_rel:+.1f}%',
    })

lift_df = pd.DataFrame(results)
print('Observed Lift (A/B Result — not yet statistically validated):')
lift_df

Observed Lift (A/B Result — not yet statistically validated):


,offer_type,control_rate,treatment_rate,lift_absolute,lift_relative
0,bogo,50.3%,67.3%,+17.0%,+33.9%
1,discount,43.2%,79.7%,+36.5%,+84.6%
2,informational,0.0%,0.0%,+0.0%,+nan%


## 5. A/B Test Summary

| Offer Type | Control Rate | Treatment Rate | Raw Lift |
|---|---|---|---|
| BOGO | 50.3% | 67.3% | +33.9% |
| Discount | 43.2% | 79.7% | +84.6% |
| Informational | — | — | N/A (no completion events) |

**Observations:**
- Discount produces a much larger raw lift than BOGO (+84.6% vs +33.9%)
- BOGO's control rate is already higher (50.3%) because the offer is more generous — customers complete it even without full awareness
- Both groups have sufficient sample size (well above the ~800 per group required for 80% power at 5pp MDE)

---
**Next:** Notebook 03 applies formal statistical tests to confirm these differences are not due to random chance.
